# FastAPI Interview Handbook — Complete (Beginner to Senior)

An interview-focused FastAPI guide for Python, backend, GenAI, RAG, and agentic-AI roles. Every section gives you the answer to say, a compact example, a production scenario, follow-ups, senior-level guidance, and pitfalls.

## How to use this handbook

Read each answer aloud, then explain the example without looking. For senior interviews, prioritize trade-offs, failure modes, observability, security, and scalability over framework definitions.

## Complete roadmap

| Module | Focus |
|---:|---|
| 1 | Fundamentals: FastAPI, ASGI, Uvicorn, Starlette, Pydantic, lifecycle |
| 2 | Routing and request handling |
| 3 | Pydantic models, validation, serialization |
| 4 | Dependency injection |
| 5 | Async programming and background work |
| 6 | Production concerns: middleware, security, logging, errors |
| 7 | Database design and SQLAlchemy |
| 8 | Deployment and operations |
| 9 | Performance and large responses |
| 10 | GenAI, RAG, streaming, WebSockets, LangChain/LangGraph |

---

# Module 1 — FastAPI Fundamentals

## 1. What is FastAPI?

### Answer
FastAPI is a modern Python framework for API services. It builds on Starlette for ASGI/web primitives and Pydantic for typed validation and serialization. Python type hints drive request validation, response contracts, and OpenAPI documentation. It is a strong fit for typed APIs, microservices, model gateways, and I/O-heavy workloads.

```python
from fastapi import FastAPI
from pydantic import BaseModel
app = FastAPI()
class User(BaseModel): id: int; name: str
@app.get('/users/{user_id}', response_model=User)
async def get_user(user_id: int): return {'id': user_id, 'name': 'Ana'}
```

### Scenario
An AI team exposes a typed `/chat` API, authenticates users, retrieves context, and streams model output. FastAPI supplies contracts and async-friendly transport.

### Follow-ups
- Does it require async? No; it supports both `def` and `async def`.
- Does it include an ORM? No; choose one such as SQLAlchemy.

### Senior tip / mistakes
Choose it for the service’s needs, not benchmark headlines. It does not make blocking code or CPU work fast.

## 2. Why FastAPI over Flask?

### Answer
FastAPI provides typed input/output validation, generated OpenAPI docs, dependency injection, and native ASGI support as first-class features. Flask is intentionally minimal and flexible; it can achieve these capabilities through extensions and custom conventions. Choose FastAPI when API contracts, developer experience, and async I/O are important; Flask remains suitable for simple or established applications.

```python
from fastapi import FastAPI
from pydantic import BaseModel
app = FastAPI()
class CreateUser(BaseModel): email: str; display_name: str
@app.post('/users', status_code=201)
async def create_user(body: CreateUser): return body
```

### Scenario
A partner API needs predictable validation errors and accurate docs for each release. FastAPI reduces custom plumbing.

### Follow-ups
- Can Flask validate data? Yes, with libraries/custom code.
- Is Flask async? Async views exist, but Flask’s traditional deployment model is WSGI.

### Senior tip / mistakes
Do not migrate merely to add `async def`; blocking database or HTTP clients will still constrain concurrency.

## 3. FastAPI vs Django vs Flask

### Answer

| Framework | Best fit | Main strength | Trade-off |
|---|---|---|---|
| FastAPI | API-first services | Typed contracts, ASGI, OpenAPI | Bring ORM/admin choices |
| Flask | Small or highly custom apps | Simplicity and flexibility | More conventions assembled by team |
| Django | Full product backend | ORM, auth, migrations, admin | More opinionated/full-stack |

Use Django for a data-heavy back office, FastAPI for a separately scaled inference service, and Flask for a small utility or mature Flask ecosystem. Keep business logic independent from the framework.

```python
class PricingService:
    def price_for(self, sku: str) -> float: return 89.0
```

### Scenario
A Django product calls an independently deployed FastAPI LLM service. The AI service can scale and release without changing the customer-facing monolith.

### Follow-ups
- Can Django serve APIs? Yes, typically with DRF or Django Ninja.
- Can FastAPI use an ORM? Yes.

### Senior tip / mistakes
Choose per bounded context and operational ownership; do not split every route into a service.

## 4. ASGI vs WSGI

### Answer
WSGI is a traditional synchronous Python web interface. ASGI supports asynchronous applications and protocols such as WebSockets and lifespan events. With non-blocking I/O, an ASGI application can yield while waiting and let the event loop progress other requests. FastAPI is ASGI; synchronous handlers are run in a thread pool.

```python
import httpx
@app.get('/status')
async def status():
    async with httpx.AsyncClient() as client:
        r = await client.get('https://example.com/status')
    return {'code': r.status_code}
```

### Scenario
Thousands of clients consume token streams. ASGI permits concurrent waiting; a blocking SDK call inside the endpoint defeats that benefit.

### Follow-ups
- Does async help CPU work? No; use processes/workers.
- Does `await` create a thread? No; it yields control to the event loop.

### Senior tip / mistakes
Distinguish concurrency from parallelism. Never use `requests` directly in an `async def` route.

## 5. Why is FastAPI fast?

### Answer
FastAPI uses the efficient ASGI/Starlette stack and Pydantic validation. Async I/O can improve utilization when a request is waiting on databases, APIs, or streams. End-to-end latency is usually dominated by downstream services, payloads, serialization, database queries, or LLM inference—not the framework.

```python
@app.get('/health')
async def health(): return {'status': 'ok'}
```

### Scenario
An LLM endpoint has 4 seconds p95 latency, of which 3.6 seconds is provider time. Improve timeout budgets, streaming, caching, and provider capacity—not framework microbenchmarks.

### Follow-ups
- What do you measure? p50/p95/p99 latency, errors, saturation, event-loop lag.
- Should all routes be async? Only if the I/O stack is async.

### Senior tip / mistakes
Profile production-like traffic before optimizing; avoid creating new clients/connections per request.

## 6. What is Uvicorn?

### Answer
Uvicorn is an ASGI server. It accepts HTTP/WebSocket connections and runs a FastAPI ASGI application. In production, use a supervisor or orchestrator, health checks, graceful shutdown, structured logs, worker tuning, and a reverse proxy/load balancer where appropriate.

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000 --workers 4
```

### Scenario
Kubernetes routes requests to several stateless Uvicorn pods and drains a pod during shutdown so in-flight requests can finish within a deadline.

### Follow-ups
- Is Uvicorn a framework? No, it is a server.
- Are workers shared-memory? No, they are separate processes.

### Senior tip / mistakes
Set worker count from CPU/memory, traffic, and upstream connection limits—not a copied rule. Never use `--reload` in production.

## 7. What is Starlette?

### Answer
Starlette is the ASGI toolkit underlying FastAPI. It provides routing, request/response objects, middleware, WebSockets, lifespan, streaming, and background tasks. FastAPI adds type-driven validation, dependencies, and OpenAPI on top.

```python
from starlette.responses import StreamingResponse
@app.get('/events')
async def events():
    async def source(): yield 'data: ready\n\n'
    return StreamingResponse(source(), media_type='text/event-stream')
```

### Scenario
A chat API uses Starlette streaming transport while FastAPI validates requests and applies authentication dependencies.

### Follow-ups
- Can Starlette be used alone? Yes.
- Does FastAPI use Starlette middleware? Yes.

### Senior tip / mistakes
Know the stack: Uvicorn → ASGI → Starlette → FastAPI. Treat CORS and middleware ordering as security-sensitive.

## 8. What is Pydantic in FastAPI?

### Answer
Pydantic turns Python types into runtime validation and serialization. A request model validates untrusted input; a response model documents and filters output. In Pydantic v2, use `model_validate()` to create/validate a model and `model_dump()` to serialize it.

```python
from pydantic import BaseModel, Field
class Order(BaseModel):
    quantity: int = Field(gt=0, le=100)
order = Order.model_validate({'quantity': 2})
```

### Scenario
An ORM user row contains a password hash. An explicit response model prevents it from reaching the API client.

### Follow-ups
- Is it business authorization? No.
- Can it read ORM attributes? Yes, with v2 `from_attributes=True` config.

### Senior tip / mistakes
Separate API schemas from database tables. Do not place database calls inside validators.

## 9. Describe a production FastAPI architecture

### Answer
Keep transport code thin. Use routers for HTTP concerns, schemas for contracts, dependencies for request-scoped wiring, services for use cases, repositories/adapters for storage and external systems, and lifespan for shared clients. Keep replicas stateless and externalize state.

```text
app/ main.py | api/routers/ | schemas/ | dependencies/
     services/ | repositories/ | clients/ | core/
```

```python
from fastapi import APIRouter, Depends
router = APIRouter(prefix='/documents')
@router.post('/{id}/index', status_code=202)
async def index(id: str, service=Depends(get_document_service)):
    return {'job_id': await service.enqueue(id)}
```

### Scenario
Indexing a document takes minutes, so the route returns `202 Accepted` and a durable worker executes the job with retries.

### Follow-ups
- Where do shared clients live? Lifespan-managed application state.
- What is `BackgroundTasks` for? Small post-response work, not durable jobs.

### Senior tip / mistakes
Include tracing, metrics, readiness checks, timeout budgets, configuration validation, and versioning in the architecture.

## 10. Explain the request lifecycle

### Answer
Uvicorn passes the request to the ASGI application. Middleware wraps it, FastAPI matches a route, resolves dependencies, validates inputs, runs the handler, serializes against the response model, then runs cleanup for yielded dependencies and middleware. Validation errors return `422` before the route executes.

```python
from collections.abc import Generator
def get_session() -> Generator[str, None, None]:
    try: yield 'session'
    finally: pass  # close/rollback a real session
@app.get('/orders/{id}')
async def order(id: int, session=Depends(get_session)): return {'id': id}
```

### Scenario
Middleware assigns a correlation ID; auth and tenant dependencies run; the service is called; the response model removes internal fields; cleanup returns a database connection.

### Follow-ups
- Are dependencies cached? Usually once per request by default.
- Where do cross-cutting concerns belong? Middleware; request-scoped policies belong in dependencies.

### Senior tip / mistakes
Avoid opening expensive clients per request, leaking internal exception text, or assuming background tasks are durable.

---

# Module 2 — Routing and Request Handling

## 11. Path parameters

### Answer
Path parameters identify a resource and are parsed from the URL path. Type hints validate and convert them.

```python
@app.get('/users/{user_id}')
async def get_user(user_id: int): return {'id': user_id}
```

### Scenario
`/tenants/{tenant_id}/documents/{document_id}` scopes a document by tenant. Still enforce tenant authorization—URL shape is not access control.

### Follow-ups / senior tip / mistakes
Use a path parameter for resource identity, query parameters for filtering/options. Static paths such as `/users/me` must be declared before `/users/{user_id}`. Do not trust IDs merely because validation passes.

## 12. Query parameters

### Answer
Query parameters express filtering, sorting, pagination, and optional behavior. Validate bounds to prevent expensive or ambiguous requests.

```python
from fastapi import Query
@app.get('/products')
async def list_products(limit: int = Query(20, ge=1, le=100), q: str | None = None):
    return {'limit': limit, 'q': q}
```

### Scenario
A search endpoint caps `limit` at 100 and uses cursor pagination to avoid scanning huge result sets.

### Follow-ups / senior tip / mistakes
Repeated query keys can map to lists. Define sorting allow-lists; never interpolate a client-provided sort field into SQL.

## 13. Request body

### Answer
Use a Pydantic model for JSON bodies to get validation, editor support, OpenAPI schemas, and clear ownership of input contracts.

```python
class CreateTicket(BaseModel): title: str; priority: int = Field(ge=1, le=5)
@app.post('/tickets', status_code=201)
async def create_ticket(payload: CreateTicket): return payload
```

### Scenario
A public endpoint rejects malformed payloads before it opens a transaction or calls an external provider.

### Follow-ups / senior tip / mistakes
Use separate create/update/read models. Do not expose internal database columns by accepting a broad object directly.

## 14. Path vs query parameters

### Answer
Use the path for the hierarchical identity of a target (`/orders/42`) and query for optional selection or representation (`/orders?status=open&limit=20`). Avoid verbs in paths when an HTTP method describes the action.

```text
GET /documents/42
GET /documents?owner=me&cursor=abc
POST /documents/42/reindex   # explicit domain action when needed
```

### Scenario
`GET /reports/{report_id}?format=csv` identifies the report in the path and chooses a representation via query.

### Follow-ups / senior tip / mistakes
Be consistent and document semantics. Do not put sensitive tokens in query strings because they often appear in logs and browser history.

## 15. Response models

### Answer
`response_model` documents, validates, serializes, and filters the public response. It is a key boundary against accidental data exposure.

```python
class UserOut(BaseModel): id: int; email: str
@app.get('/users/{id}', response_model=UserOut)
async def user(id: int): return {'id': id, 'email': 'a@x.com', 'password_hash': 'secret'}
```

### Scenario
Internal objects gain an `is_admin` field; explicit output models prevent it becoming an accidental public field.

### Follow-ups / senior tip / mistakes
Use response models per client contract and version breaking changes. Never return an ORM object blindly.

## 16. Status codes

### Answer

| Code | Use |
|---:|---|
| 200 | Successful read/update response |
| 201 | Resource created; include `Location` when appropriate |
| 202 | Accepted for asynchronous processing |
| 204 | Success with no response body |
| 400 | Invalid request semantics |
| 401 | Missing/invalid authentication |
| 403 | Authenticated but not permitted |
| 404 | Resource not found/hidden |
| 409 | State conflict, such as duplicate or version conflict |
| 422 | Parsed request fails validation |

```python
@app.post('/imports', status_code=202)
async def import_file(): return {'job_id': 'j_123'}
```

### Scenario
Large ingestion is queued and returns 202, not 200, because completion is not yet true.

### Follow-ups / senior tip / mistakes
Make error bodies stable and machine-readable. Do not use 200 for every result or leak whether a protected resource exists.

## 17. Headers

### Answer
Headers carry metadata such as authorization, request IDs, content negotiation, and conditional request values.

```python
from fastapi import Header
@app.get('/whoami')
async def whoami(x_request_id: str | None = Header(None)):
    return {'request_id': x_request_id}
```

### Scenario
An API propagates `X-Request-ID` to logs and downstream calls for tracing.

### Follow-ups / senior tip / mistakes
Treat headers as untrusted. Do not log authorization/cookie headers; enforce trusted proxy rules before accepting forwarded client IP headers.

## 18. Cookies

### Answer
Cookies can maintain browser session state. Secure cookies should use `HttpOnly`, `Secure`, an appropriate `SameSite` policy, and CSRF protection when cookies authenticate state-changing browser requests.

```python
from fastapi import Response
@app.post('/session')
async def login(response: Response):
    response.set_cookie('session', 'opaque-id', httponly=True, secure=True, samesite='lax')
    return {'ok': True}
```

### Scenario
A browser app uses session cookies; the backend uses CSRF tokens for POST/PUT/DELETE requests.

### Follow-ups / senior tip / mistakes
Cookies are automatically sent by browsers; Bearer tokens are normally explicit headers. Never put raw sensitive session data in client-readable cookies.

## 19. File uploads

### Answer
Use `UploadFile` for streamed file handling rather than loading the entire file into memory. Validate size, content type, file signature, authorization, virus-scanning workflow, and storage naming.

```python
from fastapi import File, UploadFile
@app.post('/files')
async def upload(file: UploadFile = File(...)):
    chunk = await file.read(1024)
    return {'name': file.filename, 'first_bytes': len(chunk)}
```

### Scenario
A document-ingestion service writes uploads to private object storage, queues malware scan/indexing, and returns a job ID.

### Follow-ups / senior tip / mistakes
Do not trust filename or declared MIME type. Do not use background in-memory processing for large, durable ingestion jobs.

## 20. Form data

### Answer
Use `Form` for HTML form or OAuth-style URL-encoded/multipart inputs. JSON request bodies and form data have different content types.

```python
from fastapi import Form
@app.post('/feedback')
async def feedback(message: str = Form(...)): return {'received': message}
```

### Scenario
An OAuth token endpoint receives credentials as form data per the protocol.

### Follow-ups / senior tip / mistakes
Use HTTPS; avoid logging secrets submitted in forms; distinguish a browser form endpoint from a JSON API.

---

# Module 3 — Pydantic and Data Contracts

## 21. `BaseModel` and validation

### Answer
Subclass `BaseModel` to express a schema. Pydantic parses input, validates constraints, and gives a typed object to application code.

```python
class Address(BaseModel): city: str; postal_code: str
class Customer(BaseModel): name: str; address: Address
```

### Scenario
A checkout request must either be structurally valid or fail before fulfillment logic runs.

### Follow-ups / senior tip / mistakes
Validate boundary shape here; reserve business rules such as credit availability for the service layer.

## 22. `Field()` constraints and metadata

### Answer
`Field` adds constraints, defaults, descriptions, and schema metadata.

```python
class SearchRequest(BaseModel):
    query: str = Field(min_length=1, max_length=500)
    top_k: int = Field(default=5, ge=1, le=20)
```

### Scenario
Limit prompt and retrieval sizes to control cost and latency.

### Follow-ups / senior tip / mistakes
Constraints are part of your public API. Tighten them deliberately and document changes.

## 23. Optional fields and defaults

### Answer
`str | None = None` means a field may be absent/null. A default value means it may be absent, but it will be populated. In Pydantic v2, an annotation alone does not imply a default of `None`.

```python
class UpdateProfile(BaseModel):
    display_name: str | None = None
    marketing_opt_in: bool | None = None
```

### Scenario
PATCH-like updates distinguish omitted fields from values explicitly sent as `null` where domain semantics require it.

### Follow-ups / senior tip / mistakes
Use `model_dump(exclude_unset=True)` for partial update payloads. Do not overwrite stored values with defaults unintentionally.

## 24. Nested models and collections

### Answer
Nested models make complex payloads explicit and recursively validated.

```python
class Line(BaseModel): sku: str; quantity: int = Field(gt=0)
class Cart(BaseModel): lines: list[Line]
```

### Scenario
An order API rejects one malformed line item with a precise field path in the validation response.

### Follow-ups / senior tip / mistakes
Set size bounds for user-controlled lists. Avoid deeply recursive unbounded payloads.

## 25. Field and model validators

### Answer
Field validators normalize/check one field; model validators enforce relationships among fields. Keep them deterministic, fast, and side-effect-free.

```python
from pydantic import field_validator
class EmailInput(BaseModel):
    email: str
    @field_validator('email')
    @classmethod
    def normalize(cls, v: str): return v.strip().lower()
```

### Scenario
A date-range input validates that the end is after the start before querying analytics.

### Follow-ups / senior tip / mistakes
Avoid database calls, remote calls, and authorization checks in validators; they complicate errors and performance.

## 26. `model_dump()` and serialization

### Answer
`model_dump()` creates a Python dictionary; `model_dump_json()` serializes JSON. Use flags such as `exclude_none`, `exclude_unset`, and `exclude_defaults` intentionally.

```python
data = UpdateProfile(display_name='A').model_dump(exclude_unset=True)
```

### Scenario
Only submitted profile fields are forwarded to the update service.

### Follow-ups / senior tip / mistakes
Understand difference between unset and null. Do not use serialization flags to hide unclear contract design.

## 27. `model_validate()` and ORM conversion

### Answer
`model_validate()` validates data from a dictionary or object. For ORM attribute access, configure `from_attributes=True`.

```python
from pydantic import ConfigDict
class UserOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int; email: str
```

### Scenario
A repository returns ORM entities; the API maps them explicitly to a public output schema.

### Follow-ups / senior tip / mistakes
Mapping should not cause accidental lazy-load query storms; control database loading strategy.

## 28. Serialization and custom types

### Answer
Serialization translates Python types such as datetimes, UUIDs, enums, and decimals into a stable public representation. Define conventions—usually ISO 8601 UTC timestamps and string UUIDs.

```python
from datetime import datetime, timezone
class Event(BaseModel): created_at: datetime
Event(created_at=datetime.now(timezone.utc)).model_dump(mode='json')
```

### Scenario
Mobile and browser clients require consistent UTC time handling across regions.

### Follow-ups / senior tip / mistakes
Never return naive datetimes. Document precision and backward compatibility for date/time formats.

## 29. Union/discriminated models

### Answer
Use discriminated unions when a payload can take known different shapes. A discriminator makes parsing and generated schemas unambiguous.

```python
from typing import Literal, Union
class TextPart(BaseModel): type: Literal['text']; text: str
class ImagePart(BaseModel): type: Literal['image']; url: str
Part = Union[TextPart, ImagePart]
```

### Scenario
A multimodal chat endpoint accepts text or image message parts.

### Follow-ups / senior tip / mistakes
Prefer an explicit discriminator over ambiguous broad unions; version variants carefully.

## 30. Pydantic security boundaries

### Answer
Pydantic validates shape but does not automatically prevent injection, enforce ownership, escape HTML, or secure secrets. Combine schemas with parameterized queries, authorization checks, output encoding, and secret management.

```python
class PublicUser(BaseModel): id: int; name: str
class InternalUser(PublicUser): password_hash: str
```

### Scenario
The response is always built as `PublicUser`, even though the service internally uses `InternalUser`.

### Follow-ups / senior tip / mistakes
Use allow-list output schemas; never assume client input is safe just because its JSON shape passed validation.

---

# Module 4 — Dependency Injection

## 31. What is `Depends()`?

### Answer
`Depends()` declares a dependency that FastAPI resolves before the route. It makes shared concerns composable, testable, and explicit in the route signature.

```python
from fastapi import Depends
def get_current_user(): return {'id': 'u1'}
@app.get('/me')
async def me(user=Depends(get_current_user)): return user
```

### Scenario
Every protected endpoint receives a verified identity without duplicating token parsing.

### Follow-ups / senior tip / mistakes
Dependencies can depend on dependencies. Keep them focused; do not turn one “god dependency” into the whole application.

## 32. Authentication and authorization dependencies

### Answer
Authentication establishes who the caller is; authorization establishes what they may do. Use a dependency to parse/verify credentials, then service-level or policy checks to enforce action/resource permissions.

```python
from fastapi import HTTPException, status
def require_user(token: str = Depends(oauth2_scheme)):
    if token != 'valid': raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED)
    return {'sub': 'u1'}
```

### Scenario
An authenticated user requests a document; the service verifies document tenant ownership before returning it.

### Follow-ups / senior tip / mistakes
401 is authentication failure; 403 is authenticated but forbidden. Never authorize based only on a path ID.

## 33. Database session dependencies

### Answer
Yield a session per request so it is reliably closed/rolled back after use. The exact pattern depends on sync or async SQLAlchemy.

```python
def get_db():
    db = SessionLocal()
    try: yield db
    finally: db.close()
```

### Scenario
Each request borrows a pooled connection through its session and releases it even on errors.

### Follow-ups / senior tip / mistakes
Do not create an engine per request. Keep transactions explicit; avoid holding transactions open during slow network/LLM calls.

## 34. Reusable router-level dependencies

### Answer
Dependencies can apply to a router or application, useful for baseline authentication, request IDs, or tenant context.

```python
router = APIRouter(prefix='/admin', dependencies=[Depends(require_admin)])
```

### Scenario
All administrative endpoints require the same role baseline, while individual routes add resource-specific checks.

### Follow-ups / senior tip / mistakes
Ensure public endpoints are not accidentally included in a protected router and vice versa.

## 35. Dependency caching and `use_cache`

### Answer
FastAPI caches a dependency result per request by default, avoiding repeated work across dependent components. Set `use_cache=False` only when a fresh invocation is required.

```python
user = Depends(get_current_user)  # one resolved user per request
```

### Scenario
Token parsing is done once even though audit logging and authorization both need the current principal.

### Follow-ups / senior tip / mistakes
Do not mistake per-request caching for global caching; do not cache mutable tenant data globally without invalidation.

## 36. Testing dependency overrides

### Answer
Dependency overrides let tests replace real databases, identity providers, or external clients with controlled fakes.

```python
app.dependency_overrides[get_current_user] = lambda: {'id': 'test-user'}
```

### Scenario
Integration tests run against an isolated test database and never call a real payment service.

### Follow-ups / senior tip / mistakes
Clear overrides after tests to prevent leakage. Test important authorization policies with realistic claims.

## 37. Lifespan vs dependencies

### Answer
Use lifespan for application-scoped resources such as an HTTP client, model client, or connection pool. Use dependencies for request-scoped access and cleanup.

```python
from contextlib import asynccontextmanager
@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.client = httpx.AsyncClient()
    yield
    await app.state.client.aclose()
```

### Scenario
One reusable outbound HTTP client maintains connection pooling across requests.

### Follow-ups / senior tip / mistakes
Avoid deprecated startup/shutdown event patterns in new code where lifespan is clearer. Never store per-request data in application state.

---

# Module 5 — Async FastAPI

## 38. `async def` vs `def`

### Answer
Use `async def` when the route awaits truly async I/O (async DB, HTTP, queues). Use normal `def` for blocking libraries; FastAPI runs it in a thread pool. Do not call blocking functions directly inside `async def`.

```python
@app.get('/async')
async def async_route(): return await async_repository.fetch()
@app.get('/sync')
def sync_route(): return blocking_repository.fetch()
```

### Scenario
A legacy synchronous SDK belongs in a synchronous route or is offloaded, while an `httpx.AsyncClient` call belongs in an async route.

### Follow-ups / senior tip / mistakes
Async is not automatically faster. A thread pool can exhaust under blocking load; measure and isolate work.

## 39. Event loop and `await`

### Answer
The event loop schedules coroutine progress. `await` suspends the current coroutine until an awaitable completes, allowing other tasks to run; it does not make a blocking function non-blocking.

```python
async def fetch_both(a, b):
    return await asyncio.gather(a(), b())
```

### Scenario
Two independent slow provider calls can be concurrent, subject to rate limits and cancellation rules.

### Follow-ups / senior tip / mistakes
Apply explicit timeouts and concurrency limits. Unbounded `gather()` can overload downstream services.

## 40. Blocking I/O and CPU-bound work

### Answer
Blocking I/O blocks an event-loop thread; use async clients or a controlled thread offload. CPU-bound work occupies the interpreter and should run in processes, dedicated worker services, or specialized infrastructure.

```python
from starlette.concurrency import run_in_threadpool
@app.get('/legacy')
async def legacy(): return await run_in_threadpool(blocking_library_call)
```

### Scenario
PDF parsing and large embedding generation are queued to workers rather than performed in an HTTP process.

### Follow-ups / senior tip / mistakes
Offloading is not infinite capacity. Do not run unrestricted CPU tasks in a request worker.

## 41. Async HTTP and connection reuse

### Answer
Create one reusable `httpx.AsyncClient` at lifespan, set timeouts, and use pooling. Per-request clients prevent pooling and increase socket churn.

```python
timeout = httpx.Timeout(10.0, connect=2.0)
client = httpx.AsyncClient(timeout=timeout)
```

### Scenario
An LLM gateway limits outbound connections and gives each upstream call a connect/read/overall budget.

### Follow-ups / senior tip / mistakes
Use retries only for safe/transient cases and include jitter; propagate cancellation where supported.

## 42. BackgroundTasks

### Answer
`BackgroundTasks` runs small work after returning a response in the same process. It is convenient for low-risk notifications or cleanup, but it is not durable, independently scalable, or reliable through a process crash.

```python
from fastapi import BackgroundTasks
@app.post('/signup', status_code=202)
async def signup(tasks: BackgroundTasks):
    tasks.add_task(send_welcome_email)
    return {'accepted': True}
```

### Scenario
A welcome email may be acceptable as best effort; invoice generation needs a durable queue and worker.

### Follow-ups / senior tip / mistakes
Use Celery, Dramatiq, cloud queues, or workflow systems for durable jobs. Do not perform heavy indexing through `BackgroundTasks`.

## 43. Cancellation, timeouts, and resource limits

### Answer
Every remote call needs a timeout budget. Respect client disconnect/cancellation where possible, bound request size and concurrency, and release resources in `finally` blocks.

```python
try:
    async with asyncio.timeout(15):
        return await provider.generate()
except TimeoutError:
    raise HTTPException(504, 'upstream timeout')
```

### Scenario
If a browser disconnects during an expensive streaming request, the service cancels downstream generation if the provider supports it.

### Follow-ups / senior tip / mistakes
Use 504 for upstream timeout/gateway semantics. Do not retry indefinitely or hide cancellations.

---

# Module 6 — Production FastAPI

## 44. Middleware

### Answer
Middleware wraps requests/responses for cross-cutting concerns such as correlation IDs, timing, CORS, compression, and security headers. Keep it thin; business authorization belongs in dependencies/services.

```python
@app.middleware('http')
async def request_id(request, call_next):
    request_id = request.headers.get('X-Request-ID', str(uuid4()))
    response = await call_next(request)
    response.headers['X-Request-ID'] = request_id
    return response
```

### Scenario
Every log, trace, and response shares a correlation ID for support investigations.

### Follow-ups / senior tip / mistakes
Middleware order matters. Do not read/consume a streaming request body casually in middleware.

## 45. CORS

### Answer
CORS is a browser policy that controls cross-origin JavaScript access. Configure known origins, methods, and headers; it is not an API authentication mechanism.

```python
from fastapi.middleware.cors import CORSMiddleware
app.add_middleware(CORSMiddleware, allow_origins=['https://app.example.com'],
                   allow_credentials=True, allow_methods=['GET','POST'],
                   allow_headers=['Authorization','Content-Type'])
```

### Scenario
A browser frontend at `app.example.com` calls an API at `api.example.com` with credentials.

### Follow-ups / senior tip / mistakes
Wildcard origin cannot be safely combined with credentials. CORS does not protect non-browser callers.

## 46. Exception handling

### Answer
Use domain exceptions and centralized exception handlers to produce stable, safe error envelopes. Log rich internal context server-side, but do not expose stack traces or secrets to clients.

```python
from fastapi import Request
from fastapi.responses import JSONResponse
class NotFoundError(Exception): pass
@app.exception_handler(NotFoundError)
async def not_found(_: Request, exc: NotFoundError):
    return JSONResponse(404, {'code': 'not_found', 'message': 'Resource not found'})
```

### Scenario
A missing tenant record consistently returns a documented `not_found` error across routes.

### Follow-ups / senior tip / mistakes
Differentiate client, authorization, conflict, and upstream failures. Do not catch every exception and return 200.

## 47. Logging, metrics, tracing

### Answer
Use structured logs with request IDs, safe principal/tenant identifiers, route, latency, and error code. Add RED metrics (rate, errors, duration) and distributed tracing across dependencies. Redact secrets and sensitive content.

```python
logger.info('request_complete', extra={'request_id': rid, 'route': route, 'status': 200})
```

### Scenario
An LLM timeout is correlated from browser request through retrieval, model gateway, and provider call.

### Follow-ups / senior tip / mistakes
Log metadata rather than full prompts by default. Metrics should be low-cardinality; do not label every user ID.

## 48. OAuth2 and JWT

### Answer
OAuth2 is an authorization framework; JWT is a token format. Verify token signature, issuer, audience, expiry, allowed algorithms, and relevant claims. Prefer a proven identity provider; avoid implementing token cryptography yourself.

```python
# Pseudocode: verify signature and claims with provider JWKS before trusting sub/scopes.
claims = verify_jwt(token, issuer=ISSUER, audience=API_AUDIENCE)
```

### Scenario
An API accepts access tokens from an enterprise IdP, checks scopes, and authorizes resource access by tenant.

### Follow-ups / senior tip / mistakes
JWTs are not automatically revocable; design expiry and revocation/session strategy. Never decode without verification.

## 49. Rate limiting and abuse controls

### Answer
Rate-limit at the edge or shared store by trusted client/API key/user/tenant. Apply quotas, payload limits, concurrency limits, and cost limits; return `429` with retry guidance where appropriate.

```text
Key: tenant_id + route; policy: 60 requests/minute and 5 concurrent generations
```

### Scenario
An expensive generation endpoint has per-tenant token budgets and concurrency caps to avoid runaway bills.

### Follow-ups / senior tip / mistakes
In-memory rate limits fail across replicas. Be careful with client IP behind proxies and NAT.

## 50. Configuration and secrets

### Answer
Use typed settings, injected from environment/secrets management, validated at startup. Separate environments and rotate credentials. Never commit secrets or log them.

```python
from pydantic_settings import BaseSettings
class Settings(BaseSettings): database_url: str; environment: str = 'dev'
```

### Scenario
Production database credentials are mounted from a managed secret store; local development uses a non-production `.env` file excluded from source control.

### Follow-ups / senior tip / mistakes
Fail fast for required settings. Do not use production defaults for development or vice versa.

---

# Module 7 — Databases

## 51. SQLAlchemy session and engine lifecycle

### Answer
The engine and its connection pool are application-scoped; sessions are unit-of-work/request scoped. Create engine once, acquire sessions via a dependency, commit/rollback deliberately, and close sessions reliably.

```python
engine = create_engine(settings.database_url, pool_pre_ping=True)
SessionLocal = sessionmaker(bind=engine)
```

### Scenario
Each API request gets a session while the engine reuses managed DB connections.

### Follow-ups / senior tip / mistakes
Pool size must account for all workers/replicas. Do not create engines inside routes.

## 52. Async SQLAlchemy

### Answer
Use SQLAlchemy’s async engine/session with an async database driver when the request path is async. Await database operations and understand driver compatibility.

```python
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker
engine = create_async_engine('postgresql+asyncpg://...')
AsyncSessionLocal = async_sessionmaker(engine)
```

### Scenario
An API handles many concurrent, short database reads without blocking the event loop.

### Follow-ups / senior tip / mistakes
Async is not a substitute for indexing/query optimization. Avoid triggering implicit lazy loads in response serialization.

## 53. Transactions and consistency

### Answer
A transaction groups database operations so they commit atomically or roll back. Define transaction boundaries around the business operation and choose isolation/idempotency strategies for concurrent requests.

```python
async with session.begin():
    session.add(order)
    session.add(audit_event)
```

### Scenario
Creating an order and reserving inventory must succeed together. External payment calls are handled with idempotency/outbox patterns, not an open DB transaction around a network call.

### Follow-ups / senior tip / mistakes
Keep transactions short. Handle unique constraint failures as conflicts where appropriate.

## 54. N+1 queries and loading strategy

### Answer
N+1 happens when one query fetches a list and then one additional query loads related data per item. Use joins/eager loading, batch queries, and page sizes appropriate to the response.

```python
stmt = select(Order).options(selectinload(Order.lines)).limit(50)
```

### Scenario
An orders listing with 50 records must not execute 51 SQL queries to display line counts.

### Follow-ups / senior tip / mistakes
Measure query count/latency in tests or tracing. Do not eager-load every relationship indiscriminately.

## 55. Migrations and schema evolution

### Answer
Use migrations (commonly Alembic) for versioned, reviewable schema changes. Deploy backward-compatible changes first, then application changes, then cleanup in a later release.

```text
1. Add nullable column / new table
2. Deploy code that writes both forms
3. Backfill and verify
4. Enforce constraint; remove old path later
```

### Scenario
Adding tenant IDs to a large table is performed in stages to avoid locking or downtime.

### Follow-ups / senior tip / mistakes
Test migrations on production-like data volume. Never depend on ORM auto-create in production.

---

# Module 8 — Deployment and Operations

## 56. Dockerizing FastAPI

### Answer
Build a small reproducible image, pin dependencies, run as a non-root user, expose a health endpoint, pass configuration at runtime, and avoid embedding secrets in layers.

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Scenario
The same image is promoted from test to production; only config and infrastructure bindings differ.

### Follow-ups / senior tip / mistakes
Use multi-stage builds if native compilation is needed. Do not bake `.env` or private keys into images.

## 57. Gunicorn, Uvicorn workers, and process model

### Answer
Uvicorn serves ASGI. Gunicorn can manage multiple Uvicorn worker processes in some deployment models. Containers/orchestrators can also manage one process per container and scale replicas. Choose one clear supervision/scaling model.

```bash
gunicorn app.main:app -k uvicorn.workers.UvicornWorker --workers 4
```

### Scenario
On a VM, Gunicorn supervises workers. On Kubernetes, an organization may use one Uvicorn process per container and scale pods.

### Follow-ups / senior tip / mistakes
Workers multiply DB connections and model-client pools. Do not rely on worker-local memory for shared state.

## 58. Reverse proxies and TLS

### Answer
A load balancer/reverse proxy commonly terminates TLS, handles routing, limits request size, and forwards trusted headers. Configure the app to trust forwarded headers only from known proxy networks.

```text
Client → CDN/WAF → Load balancer / Nginx → Uvicorn/FastAPI
```

### Scenario
HTTPS terminates at the load balancer; the app receives traffic internally and correctly generates HTTPS URLs using trusted proxy headers.

### Follow-ups / senior tip / mistakes
Untrusted `X-Forwarded-For` can spoof client IP. Do not expose debug endpoints through the proxy.

## 59. Health, readiness, and graceful shutdown

### Answer
Liveness says the process should be restarted; readiness says it can receive traffic. During shutdown, stop accepting new work, let in-flight work drain within a deadline, and close shared resources.

```python
@app.get('/livez')
async def livez(): return {'ok': True}
@app.get('/readyz')
async def readyz(): return {'ok': await can_reach_required_dependency()}
```

### Scenario
A new pod is not added to the load balancer until critical dependency connectivity is ready.

### Follow-ups / senior tip / mistakes
Keep liveness checks cheap; do not restart a healthy process because a downstream optional service is briefly unavailable.

## 60. Scaling and state

### Answer
Scale stateless API replicas horizontally. Store sessions, caches, jobs, rate-limit counters, files, and shared coordination in appropriate external systems. Apply connection pool and concurrency budgets globally.

```text
FastAPI replicas ↔ shared database / Redis / object storage / queue
```

### Scenario
WebSocket/chat reconnects route safely because session and conversation state are in shared storage, not one pod’s memory.

### Follow-ups / senior tip / mistakes
Autoscaling needs meaningful signals such as queue depth, concurrency, latency, and CPU—not CPU alone for I/O-heavy APIs.

---

# Module 9 — Performance and API Design

## 61. Connection pooling

### Answer
Pools reuse database and HTTP connections, lowering handshake cost and controlling resource usage. Configure pool limits with the total number of processes/replicas in mind.

```python
engine = create_engine(url, pool_size=10, max_overflow=10, pool_timeout=30)
```

### Scenario
Ten replicas with four workers and a pool of 20 can potentially create 800 DB connections—often too many.

### Follow-ups / senior tip / mistakes
Treat pool sizing as a system-wide capacity decision, not a local default.

## 62. Caching

### Answer
Cache data that is expensive, safe to reuse, and has a clear invalidation/freshness model. Common layers include CDN, reverse proxy, Redis, and application cache. Use cache keys that include tenant, authorization scope, version, and relevant query dimensions.

```text
Key: search:v3:tenant-42:query-hash:filters-hash
```

### Scenario
Common read-only product metadata uses a short shared cache; personalized answers are cached only with tenant/user isolation.

### Follow-ups / senior tip / mistakes
Cache invalidation and authorization leakage are the main risks. Never use a global key for a tenant-scoped response.

## 63. Streaming responses and SSE

### Answer
Streaming sends partial output as it becomes available, improving perceived latency and reducing buffering. SSE is a simple server-to-browser unidirectional event protocol over HTTP; handle disconnects, heartbeats, errors, and proxy buffering.

```python
from fastapi.responses import StreamingResponse
@app.get('/stream')
async def stream():
    async def tokens():
        for token in ['Hello', ' ', 'world']:
            yield token
    return StreamingResponse(tokens(), media_type='text/plain')
```

### Scenario
An LLM endpoint begins returning tokens quickly even though total generation takes 20 seconds.

### Follow-ups / senior tip / mistakes
Do not claim completion until the provider completes. Ensure output moderation, quotas, and cancellation work with streams.

## 64. Pagination

### Answer
Offset pagination is simple but can become slow/inconsistent for deep or changing lists. Cursor/keyset pagination is typically better for large ordered datasets.

```python
@app.get('/events')
async def events(cursor: str | None = None, limit: int = Query(50, le=100)):
    return {'items': [], 'next_cursor': None}
```

### Scenario
An audit feed uses an opaque cursor based on `(created_at, id)` for stable ordering.

### Follow-ups / senior tip / mistakes
Make cursor contents tamper-resistant/opaque. Always define a deterministic sort order.

## 65. Idempotency and retries

### Answer
For operations that can be retried (payments, job creation), accept an idempotency key and store the first result keyed by caller and operation. Clients may retry network failures without causing duplicate effects.

```text
Idempotency-Key: 8c37...  → stored request fingerprint + resulting order ID
```

### Scenario
A client times out after submitting an import; retry returns the original job rather than queueing the file twice.

### Follow-ups / senior tip / mistakes
Define expiry, payload matching, concurrent key handling, and who owns the key namespace.

---

# Module 10 — GenAI and Agentic AI Integration

## 66. Streaming LLM responses

### Answer
Design streaming as a cancellable, observable pipeline: authenticate/authorize, validate request and budgets, retrieve context, invoke the provider with timeouts, stream normalized events, persist the final result, and emit a terminal success/error event.

```python
@app.post('/chat/stream')
async def chat_stream(request: ChatRequest):
    async def events():
        async for token in llm_client.stream(request.message):
            yield f'data: {token}\n\n'
    return StreamingResponse(events(), media_type='text/event-stream')
```

### Scenario
A client sees generated text immediately; the server stops provider generation on disconnect and records token usage for billing.

### Follow-ups / senior tip / mistakes
Send structured event types (`token`, `citation`, `error`, `done`) rather than ambiguous raw strings. Never log prompts or tokens indiscriminately.

## 67. Background document indexing

### Answer
Indexing is a durable asynchronous workflow: upload to object storage, create a job, enqueue work, extract text, scan/validate, chunk, embed, write vectors and metadata, and expose job state. Make every step idempotent and retryable.

```python
@app.post('/documents', status_code=202)
async def create_document(file: UploadFile):
    document_id = await storage.save(file)
    job_id = await jobs.enqueue('index_document', document_id)
    return {'document_id': document_id, 'job_id': job_id}
```

### Scenario
An embedding provider outage retries only the failed chunk batch; the completed chunk writes are not duplicated.

### Follow-ups / senior tip / mistakes
Use durable queues and status persistence. Do not run multi-minute ingestion as an HTTP request or only with `BackgroundTasks`.

## 68. RAG API design

### Answer
A RAG API needs clear tenancy and access controls across ingestion and retrieval. At query time: authenticate, validate input/budget, retrieve with metadata filters, optionally rerank, build a grounded prompt, generate, return citations, evaluate/log safely, and enforce timeouts.

```python
class AskRequest(BaseModel): question: str = Field(min_length=1, max_length=4000)
@app.post('/knowledge-bases/{kb_id}/ask')
async def ask(kb_id: str, body: AskRequest, user=Depends(require_user)):
    return await rag_service.answer(kb_id, body.question, user)
```

### Scenario
Every vector query filters by tenant and document permissions before retrieved passages reach the prompt.

### Follow-ups / senior tip / mistakes
Return provenance/citations and measure retrieval separately from generation. Never trust vector-store similarity as authorization.

## 69. File upload APIs for AI

### Answer
Secure AI uploads with authentication, quotas, file size limits, MIME and signature validation, malware scanning, private object storage, encryption, retention/deletion policies, and asynchronous processing. Preserve source/version metadata for reproducibility.

```text
upload → quarantine storage → malware scan → durable index job → searchable status
```

### Scenario
A PDF is quarantined until scanned; only then can it be embedded and exposed to retrieval.

### Follow-ups / senior tip / mistakes
Prevent zip bombs and parser attacks. Do not make user-uploaded files public by default.

## 70. WebSockets vs SSE

### Answer
Use SSE for server-to-client streaming over standard HTTP when the browser mostly receives events. Use WebSockets for bidirectional real-time interaction, such as client control messages, collaborative sessions, or live tool progress. Both need authentication, authorization, connection limits, heartbeats, backpressure, and disconnect cleanup.

```python
from fastapi import WebSocket
@app.websocket('/ws/chat')
async def ws_chat(ws: WebSocket):
    await ws.accept()
    await ws.send_json({'type': 'ready'})
```

### Scenario
SSE streams LLM tokens; WebSockets support a live agent UI where the user can interrupt or answer clarification prompts.

### Follow-ups / senior tip / mistakes
Do not hold unbounded per-connection state in memory. Authenticate the handshake; validate every incoming message.

## 71. LangChain integration

### Answer
Treat LangChain as an integration/orchestration library, not your application architecture. Isolate it behind a service interface, pin versions, pass request context safely, set timeouts, and instrument every external model/tool/retriever call.

```python
class AnswerService:
    async def answer(self, question: str, tenant_id: str) -> Answer: ...
```

### Scenario
Routes depend on `AnswerService`; the service may use LangChain internally while API schemas and observability stay stable if libraries change.

### Follow-ups / senior tip / mistakes
Avoid leaking framework-specific chain objects into routes. Test prompts/retrieval behavior with fixtures and evaluation sets.

## 72. LangGraph and long-running agents

### Answer
Use LangGraph or a workflow engine when an agent has explicit state, branches, tool calls, checkpoints, human approval, retries, or resumability. Persist state outside the FastAPI process; the API starts/runs/observes a workflow rather than holding an HTTP request open indefinitely.

```text
POST /agent-runs → 202 + run_id
GET /agent-runs/{run_id} → state/events/result
POST /agent-runs/{run_id}/approve → resume transition
```

### Scenario
An agent preparing a report pauses for a human approval before sending an external action, survives deployment restarts, and continues from a checkpoint.

### Follow-ups / senior tip / mistakes
Model tool permissions and human approval as workflow states. Do not assume agent retries are safe without idempotent tools.

## 73. LLM tool calling and security

### Answer
The model proposes tool calls; the application remains the policy enforcement point. Validate structured arguments, authorize against the actual user/tenant, enforce allow-lists, limit side effects, use idempotency, and record audits. Treat retrieved content and tool output as untrusted for prompt-injection defense.

```python
def execute_tool(call, principal):
    validate_schema(call.arguments)
    authorize(principal, call.name, call.arguments)
    return tool_registry[call.name](**call.arguments)
```

### Scenario
A model suggests `delete_document`; the server requires an explicit authorized principal and a confirmation workflow before the actual deletion tool runs.

### Follow-ups / senior tip / mistakes
Never give model output direct shell/database/network authority. Apply least privilege and audit tool use.

## 74. Evaluations, observability, and cost control

### Answer
Operate GenAI endpoints with quality evaluations, safety checks, tracing, token/cost accounting, cache strategy, latency budgets, model fallbacks, and rate/concurrency limits. Measure retrieval relevance, groundedness, tool success, user outcomes, and unsafe/error rates—not only HTTP uptime.

```text
Trace: request → retrieval → rerank → prompt build → model → tools → response
Metrics: p95 latency, tokens, cost, retrieval hit rate, abstention, error rate
```

### Scenario
An answer quality regression is detected by an offline golden set and production feedback, then traced to a changed chunking configuration.

### Follow-ups / senior tip / mistakes
Version prompts, models, embeddings, index configuration, and evaluation data. Do not silently change a production prompt without measuring impact.

---

# Senior interview rapid-fire answers

| Question | Strong short answer |
|---|---|
| Why FastAPI? | Typed contracts, generated OpenAPI, ASGI, composable dependencies, good fit for API-first/I/O-heavy services. |
| Does async make it faster? | It improves concurrent I/O utilization; it does not accelerate CPU-bound work or blocking libraries. |
| How do you protect an API? | Authentication, resource authorization, secure configuration, rate/size limits, validation, safe errors, audit logs, and edge protections. |
| How do you run long work? | Return 202, persist a job, queue durable work, expose status/events, and make steps idempotent. |
| How do you scale? | Stateless replicas, shared external state, controlled pools/concurrency, observability-driven autoscaling. |
| Biggest RAG security risk? | Retrieval and vector similarity are not authorization; enforce tenant/document permissions before prompt construction. |
| How do you debug latency? | Trace the request, compare p50/p95/p99 across each dependency, inspect saturation, then fix the dominant bottleneck. |

# Final revision checklist

- Explain Uvicorn, ASGI, Starlette, FastAPI, and Pydantic as distinct layers.
- Defend `async def` versus `def` with a real dependency example.
- Describe typed request/response boundaries and why they reduce data leakage.
- Explain dependency injection, lifespan, DB session lifecycle, and testing overrides.
- Design a secure long-running file/RAG indexing flow with `202` and durable jobs.
- Describe production controls: timeouts, retries, pooling, health checks, logs, metrics, traces, CORS, JWT verification, rate limits, and secret management.
- Explain RAG tenancy, streaming, tool-call authorization, evaluation, and cost controls.

